# 006 — Prep Item2Vec (MovieLens)

Port of the reference `021-prep-item2vec.ipynb`. Build per-user movie-id
sequences (chronological, length > 1) from the feature parquets and persist
them as JSONL for item2vec / sequence-model training, plus a 2-sequence
overfit batch.

In [1]:
import json
import os

import pandas as pd
from dotenv import load_dotenv
from loguru import logger
from pydantic import BaseModel

_PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
load_dotenv(os.path.join(_PROJECT_ROOT, ".env"))


class Args(BaseModel):
    run_name: str = "006-prep-item2vec"
    user_col: str = "userId"
    item_col: str = "movieId"
    timestamp_col: str = "event_timestamp"
    project_root: str = ""
    engineer_dir: str = ""

    def init(self):
        self.project_root = _PROJECT_ROOT
        self.engineer_dir = os.path.join(self.project_root, "feature", "output", "engineer")
        return self


args = Args().init()


def get_sequence(df, user_col, item_col, ts_col):
    return (
        df.sort_values(ts_col)
        .groupby(user_col)[item_col]
        .agg(list)
        .loc[lambda s: s.apply(len) > 1]
        .values.tolist()
    )

In [2]:
train_df = pd.read_parquet(os.path.join(args.engineer_dir, "train_features.parquet"))
val_df = pd.read_parquet(os.path.join(args.engineer_dir, "val_features.parquet"))
logger.info(f"train={train_df.shape}  val={val_df.shape}")

item_sequence = get_sequence(train_df, args.user_col, args.item_col, args.timestamp_col)
val_item_sequence = get_sequence(val_df, args.user_col, args.item_col, args.timestamp_col)
logger.info(f"train sequences={len(item_sequence):,}  val sequences={len(val_item_sequence):,}")

2026-07-16 00:44:23.462 | INFO     | __main__:<module>:3 - train=(79425, 22)  val=(103, 22)


2026-07-16 00:44:23.502 | INFO     | __main__:<module>:7 - train sequences=578  val sequences=11


In [3]:
def write_jsonl(path, sequences):
    with open(path, "w", encoding="utf-8") as f:
        for seq in sequences:
            f.write(json.dumps([int(x) for x in seq]) + "\n")

write_jsonl(os.path.join(args.engineer_dir, "train_item_sequence.jsonl"), item_sequence)
write_jsonl(os.path.join(args.engineer_dir, "val_item_sequence.jsonl"), val_item_sequence)

# 2-sequence overfit batch
write_jsonl(os.path.join(args.engineer_dir, "batch_sequences_overfit.jsonl"), item_sequence[:2])
logger.info("006 persisted OK")

2026-07-16 00:44:23.533 | INFO     | __main__:<module>:11 - 006 persisted OK
